In [ ]:
# PyTorch functions/methods helpers

# 7.1.2
edge_detector = nn.Conv2d(1, # Input channel
                          1, # Output channel
                          kernel_size=(1, 2), # Kernel height * width, or inspect 1 row and 2 neighboring columns at a time
                          bias=False)

with torch.no_grad():
    edge_detector.weight[:] = torch.tensor([[[1.0 , -1.0]]]) # nn.Conv2d initialize params at random values by default, we change it to [1.0, -1.0]

# 7.1.5
torch.manual_seed(0)
flat = image.flatten()
permutation = torch.randperm(flat.numel()) # Generate a random ordering of all valid indices in flat. Since flat has 25 elements, this is a random ordering of 0–24
shuffled = flat[permutation].reshape_as(image) # Reorder the values using the random permutation, then reshape them back to the image's original shape (1, 1, 5, 5)

* Chapter 7 begins with a modeling question: what changes when the input is not just a feature vector, but a spatial object?

* CNNs are built from assumptions about images and other grid-like data: **nearby values are related, useful local patterns repeat across positions, and channels store different measurements at the same location**.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain why CNNs are an inductive bias rather than just another layer type
- explain why flattening images too early throws away spatial structure
- distinguish translation equivariance from invariance in plain English
- compare MLP and convolution parameter counts
- connect locality and weight sharing to the convolution layer

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

# 7.1.0 The Problem This Notebook Solves

An MLP treats an input as a vector. That is appropriate when the feature order has no special geometry or when the model has enough data and capacity to learn all relationships from scratch.

Images are different. Pixel location matters, and nearby pixels usually form meaningful local patterns such as edges, corners, strokes, textures, and object parts.

If we flatten an image immediately, the model can still learn, but we have hidden useful structure:

```text
image grid: row, column, channel relationships are explicit
flattened vector: location relationships must be rediscovered from index positions
```

CNNs add an inductive bias.

An inductive bias is a built-in modeling assumption that makes some solutions easier to learn than others. The CNN bias says:

- local neighborhoods are important
- the same detector can be useful at many positions
- channels are different measurements attached to each location

This is not a guarantee that CNNs are always best. It is a bet about the data. For images, that bet is often strong.

The handoff from Chapter 6 is direct:
* Chapter 6 taught modules as reusable computation blocks.
* Chapter 7 now asks what kind of block is appropriate when the input has spatial structure.

# 7.1.1 Flattening Hides Where Pixels Live

A flattened image is still data, but adjacency becomes implicit.
* Pixel 6 and pixel 7 might be neighbors in the original image, or they might be separated by a row boundary depending on the flattening convention.
* A fully connected layer can learn from flattened values, but it has to learn spatial relationships indirectly.

The cell creates a simple vertical bright region and then shifts it one column to the right.
* As images, those two patterns are obviously related.
* As flat vectors, they are just different positions in a long list.

The theoretical issue is not that flattening destroys the values. It destroys the **easy access to geometry**.
* A dense layer can assign separate weights to every position, but it does not automatically know that the same edge detector might be useful one column over.

Before running the cell, predict:

- Both tensors should have shape `(1, 1, 5, 5)`.
- The flattened nonzero indices should change after shifting.
- The shifted image is related spatially, but not identical as a flat vector.

In [ ]:
image = torch.zeros(1, 1, 5, 5)
image[:, :, :, 0:2] = 1.0 # For every batch, every channel, every row, set columns 0 and 1 to 1

# 1 1 0 0 0
# 1 1 0 0 0
# 1 1 0 0 0
# 1 1 0 0 0
# 1 1 0 0 0

shifted = torch.zeros(1, 1, 5, 5)
shifted[:, :, :, 1:3] = 1.0 # For every batch, every channel, every row, set columns 1 and 2 to 1

# 0 1 1 0 0
# 0 1 1 0 0
# 0 1 1 0 0
# 0 1 1 0 0
# 0 1 1 0 0

image_nonzero = image.flatten().nonzero().flatten()[:10]
shifted_nonzero = shifted.flatten().nonzero().flatten()[:10]

print("first nonzero flattened positions:", image_nonzero.tolist())
print("first nonzero flattened positions after shift:", shifted_nonzero.tolist())

assert shape(image) == (1, 1, 5, 5)
assert not torch.equal(image.flatten(), shifted.flatten())

first nonzero flattened positions: [0, 1, 5, 6, 10, 11, 15, 16, 20, 21]
first nonzero flattened positions after shift: [1, 2, 6, 7, 11, 12, 16, 17, 21, 22]


# 7.1.2 A Shared Local Detector Moves With the Pattern

This is the central CNN idea in miniature.

* A local detector is a small set of weights that looks at a local window.
* A shared detector means the same weights are reused at many positions.
* If the input pattern moves, the detector's response moves with it.

That behavior is called translation equivariance:

```text
shift the input -> the feature map shifts in the corresponding way
```

Equivariance is not invariance. Invariance would mean:

```text
shift the input -> the final answer stays the same
```

Convolutions give equivariant feature maps.
* Pooling, aggregation, and later classifier layers can help produce more invariant final decisions.
* Keeping that distinction clear prevents a common conceptual mistake: a **convolution does not ignore location; it preserves location while reusing a detector across locations**.

The cell uses a hand-set detector so the idea is visible before learning enters the picture.

In [ ]:
edge_detector = nn.Conv2d(1, # Input channel
                          1, # Output channel
                          kernel_size=(1, 2), # Kernel height * width, or inspect 1 row and 2 neighboring columns at a time
                          bias=False)

with torch.no_grad():
    edge_detector.weight[:] = torch.tensor([[[1.0 , -1.0]]]) # nn.Conv2d initialize params at random values by default, we change it to [1.0, -1.0]

# Apply the [1, -1] kernel across the image. For each 1x2 window, calculate: (left pixel * 1) + (right pixel * -1) = left pixel - right pixel
response = edge_detector(image)

#  0 1 0 0
#  0 1 0 0
#  0 1 0 0
#  0 1 0 0
#  0 1 0 0

shifted_response = edge_detector(shifted)

# -1 0 1 0
# -1 0 1 0
# -1 0 1 0
# -1 0 1 0
# -1 0 1 0

# Returns the column index with the highest mean (average across rows)
edge_col = int(response[0, 0].mean(dim=0).argmax()) # Highest average column is index 1
shifted_edge_col = int(shifted_response[0, 0].mean(dim=0).argmax()) # Highest average column is index 2

print("response shape:", shape(response)) # Input shape for image was defined as (1, 1, 5, 5) or 1 batch, 1 channel, 5 height (rows), 5 width (columns)
                                          # This became (1, 1, 5, 4) due to kernel size (1, 2) since nn.Conv2d was inspecting 1 row with 2 columns at a time

print("edge column:", edge_col)
print("edge column after shfit:", shifted_edge_col)

assert shape(response) == (1, 1, 5, 4)
assert shifted_edge_col == edge_col + 1

response shape: (1, 1, 5, 4)
edge column: 1
edge column after shfit: 2


# 7.1.3 Locality and Weight Sharing Reduce Parameter Count

CNNs are not only about fewer parameters, but parameter efficiency is one concrete consequence of the theory.
* A dense hidden unit connected to a 28 by 28 image has a separate weight for every pixel.
* If we want 64 hidden units, the first layer already needs many weights.
* A convolutional layer uses a different idea:

```text
learn a small local detector
reuse it at many spatial positions
learn multiple detectors as output channels
```

This encodes two assumptions:

- **locality: small neighborhoods contain useful patterns**
- **sharing: the same kind of pattern can matter in different locations**

The parameter count drops since the model no longer learns a separate detector for every absolute position. It learns detectors that scan.

The tradeoff is that we have constrained the model.
* **If absolute position matters in a way that should not be shared, this bias can hurt**.
* For ordinary image features, it usually helps.

In [ ]:
height, width = 28, 28
hidden_units = 64

mlp_weights = height * width * hidden_units # 28 * 28 * 64 (takes the image and flatten all of its pixels)
conv_weights = 6 * 1 * 5 * 5 # six 5 by 5 filters over one input channel, each filter scans local 5×5 neighbourhoods of pixels across the 28×28 image
conv_bias = 6

print("MLP first-layer weights:", mlp_weights)
print("Conv weights plus bias:", conv_weights + conv_bias)
print("ratio:", round(mlp_weights / (conv_weights + conv_bias), 1))

assert conv_weights + conv_bias < mlp_weights

MLP first-layer weights: 50176
Conv weights plus bias: 156
ratio: 321.6


# 7.1.4 Channels Store What Is Present at Each Location

A channel is a feature dimension attached to every spatial location.

For an RGB image:

```text
at row r, column c:
red value
green value
blue value
```

For a hidden CNN layer:

```text
at row r, column c:
edge-like evidence
texture-like evidence
color-like evidence
part-like evidence
```

The exact hidden channel meanings are learned, not manually named.

But the shape idea is the same: spatial position says where evidence is; channel index says what kind of evidence is stored there.

This prepares the handoff to Chapter 7.4.

A real convolution kernel does not just look across height and width. It also combines input channels.

In [ ]:
X = torch.zeros(3, 2, 2) # 3 channels each containing a 2x2 matrix
X[0] += 1.0 # Take channel 0 and add 1 to every element
X[1] += 2.0 # Take channel 1 and add 2 to every element
X[2] += 3.0 # Take channel 2 and add 3 to every element

# tensor([
#     [[1., 1.],
#      [1., 1.]],

#     [[2., 2.],
#      [2., 2.]],

#     [[3., 3.],
#      [3., 3.]]
# ])

print("channel-first shape:", shape(X))
print("values at row 0, col 0 across channels:", X[:, 0, 0]) # For all channels, look at row and column indices of 0

assert shape(X) == (3, 2, 2)
assert torch.equal(X[:, 0, 0], torch.tensor([1.0, 2.0, 3.0]))

channel-first shape: (3, 2, 2)
values at row 0, col 0 across channels: tensor([1., 2., 3.])


# 7.1.5 Break It Deliberately: Shuffle Pixels

The CNN assumption is only useful when the grid has meaning.

If pixels are randomly shuffled, nearby values in the tensor may no longer be nearby values in the original image.

This is a theory failure, not just a preprocessing error.
* The model's inductive bias says local windows matter, but the data pipeline has destroyed locality.
* A convolution can still compute something, but its local windows no longer correspond to meaningful image neighborhoods.

This is why data representation matters. The model's assumptions and the tensor layout must agree.

In [ ]:
torch.manual_seed(0)
flat = image.flatten()
permutation = torch.randperm(flat.numel()) # Generate a random ordering of all valid indices in flat. Since flat has 25 elements, this is a random ordering of 0–24
shuffled = flat[permutation].reshape_as(image) # Reorder the values using the random permutation, then reshape them back to the image's original shape (1, 1, 5, 5)

original_response = edge_detector(image)
shuffled_response = edge_detector(shuffled)

# Shuffling rearranges the elements into different spatial positions while keeping the same tensor shape. This can destroy the original spatial relationships that a convolutional filter relies on.
print("original response sum:", float(original_response.abs().sum()))
print("shuffled response sum:", float(shuffled_response.abs().sum()))
print("same response:", torch.allclose(original_response, shuffled_response))

assert not torch.allclose(original_response, shuffled_response)

original response sum: 5.0
shuffled response sum: 12.0
same response: False


/tmp/ipykernel_1524/897032595.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("original response sum:", float(original_response.abs().sum()))


# 7.1 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the chapter does not need a separate notes file.

1. What is an inductive bias, and what bias does a CNN add for image-like data?
> * An inductive bias is an assumption a model makes about the kind of patterns that are likely to exist in the data
> * A CNN assumes that local patterns matter and that the same pattern can be useful regardless of where it appears in the image

2. Why does flattening an image make spatial structure harder for an MLP to exploit?
> * Flattening removes the explicit 2D spatial arrangement of the pixels, so the MLP no longer directly knows which pixels are neighbors

3. What is the difference between equivariance and invariance?
> * Equivariance means the output changes in a predictable way when the input changes. Invariance means the output stays the same despite certain changes to the input

4. Which two constraints turn a large dense image layer into a convolution-like layer?
> * Only connect each output to a local region of the image, and reuse the same weights across different spatial locations

5. Why is parameter sharing useful for images but not automatically correct for every dataset?
> * Images often contain patterns that can appear in different locations, so sharing the same filter across locations is useful
> * Other datasets may not have this spatial structure, so forcing the same parameters to be shared could be a bad assumption

6. Why are channels best understood as measurements at the same location?
> * Because each channel usually represents a different measurement of the same spatial location
> * For example, an RGB image has red, green, and blue values for the same pixel location (3, height, width)